# Self-Refine — a multi-agent refinement loop (Reflexion-style)

[Reflexion](https://arxiv.org/abs/2303.11366) (Shinn et al.) and
[Self-Refine](https://arxiv.org/abs/2303.17651) (Madaan et al.) share one idea:
an answer gets better when a critic tells the producer *exactly* what it got
wrong and the producer tries again. This notebook rebuilds that idea as three
named agents in a closed loop — a **planner**, a tool-using **worker**, and a
devil's-advocate **debator** — that keep going until the debator runs out of
faults (or we hit an iteration cap).

The load-bearing detail: the worker **re-attempts from scratch each round**,
carrying only a *verbal* critique of its last attempt. It never edits its old
draft. That is the Reflexion property, and it's why a separate critic agent that
emits a *structured* fault list is the heart of the system.

```
                  +---- critique (typed fault list) ----+
                  v                                      |
  task --> PLANNER --plan--> WORKER --output+handoff--> DEBATOR
           output_schema     tools + handoff=            output_schema
           =Plan             (fresh attempt each round)  =Critique

  handoff shape picked ONCE by a CLASSIFIER -> Engineering | Research | Minimal
  stop when:  iterations == max_loop_iterations   OR   faults < debator_items
  final answer = latest worker output
```

**SDK primitives this notebook showcases:**
- `output_schema` + Pydantic — typed `Plan` and typed `Critique`; the critique's
  `len(items)` is the loop's deterministic stop signal (no bullet-counting).
- `handoff=` + prebuilt handoffs (`EngineeringHandoff` / `ResearchHandoff` /
  `MinimalHandoff`) — the worker emits a cold-start handoff doc describing what it did.
- `AgentInput(context_items=...)` — drops that handoff into the debator as a
  context primitive, with no string-stuffing.
- `.fork()` — spin off the planner and debator with empty history each round (the
  worker is rebuilt with its runtime-chosen handoff), so only the critique crosses rounds.
- `TokenBudgetMiddleware` / `CostBudgetMiddleware` / `RuntimeLimitMiddleware` —
  guards for a loop whose cost scales as roughly `3 x iterations`.

> **Before running:** copy `.env.example` to `.env` at the repo root and add your provider key.

## 1. Environment & constants

| Constant | What it controls |
|---|---|
| `PROVIDER` / `MODEL` | LLM for every agent. Swap to `anthropic` / `claude-opus-4-8` for a different model. |
| `MAX_LOOP_ITERATIONS` | Hard cap on plan -> execute -> critique rounds. The loop never exceeds this, no matter what. |
| `DEBATOR_ITEMS` | Stop when the debator finds **fewer than** this many faults. `1` = stop only at zero faults; `3` = stop at 0-2 faults. |
| `MAX_TOOL_ROUNDS` | How many search -> read cycles the worker may run per attempt. |
| `TOKEN_BUDGET` / `MAX_COST_PER_AGENT` / `COST_PER_MTOK` | Per-worker-attempt token and dollar caps (Section 4). `CostBudgetMiddleware` needs your model's blended \$/1M-token rate to turn tokens into dollars. |

In [ ]:
%pip install -q vidbyte-sdk python-dotenv requests pydantic

import os
from dataclasses import dataclass, field

import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display
from pydantic import BaseModel, Field

load_dotenv()
load_dotenv("../../.env")

PROVIDER = os.getenv("VIDBYTE_COOKBOOK_PROVIDER", "openai")
MODEL    = os.getenv("VIDBYTE_COOKBOOK_MODEL", "gpt-4.1")

MAX_LOOP_ITERATIONS = 3       # hard cap on plan -> execute -> critique rounds
DEBATOR_ITEMS       = 1       # stop when the debator finds FEWER than this many faults
MAX_TOOL_ROUNDS     = 6       # search -> read cycles the worker may run per attempt
TOKEN_BUDGET        = 25_000  # per-worker-attempt token cap (middleware)
MAX_COST_PER_AGENT  = 0.40    # per-worker-attempt dollar cap (middleware)
COST_PER_MTOK       = 4.0     # rough blended $ per 1M tokens for MODEL; set this for your model

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"), \
    "Set OPENAI_API_KEY (or ANTHROPIC_API_KEY + overrides) in .env first."

## 2. System prompts

Four roles, four prompts. Three are the named loop agents; the fourth is a tiny
**classifier** that decides the handoff shape once, up front.

- **Classifier** — reads the task and picks one handoff shape (`engineering` /
  `research` / `minimal`). This is how "the agent decides the handoff shape"
  becomes a concrete, cheap, deterministic step.
- **Planner** — turns the task into ordered subgoals. On later rounds it is handed
  the critique and told to restructure the plan so the next attempt can't repeat
  those faults.
- **Worker** — does the task with its tools. The key line is "you are starting
  fresh — do NOT assume any previous work exists": each round is a clean attempt
  guided only by the plan and the critique checklist.
- **Debator** — the devil's advocate. Its only job is to find faults and to return
  an *empty* list when the work is genuinely solid. That empty list is what stops
  the loop, so the prompt must discourage both sycophancy and padding.

> **Your turn:** the WORKER is told it's "starting fresh" each round. Name one task
> where feeding the worker its *previous draft* instead would clearly help — and one
> where it would make the worker stubbornly defend a bad first draft. Which kind is
> the demo task below?

In [ ]:
CLASSIFIER_PROMPT = """You pick the right handoff-document shape for a task.
Choose exactly one kind:
- "engineering" - the task produces or changes code, configs, or a concrete build artifact.
- "research"    - the task gathers, evaluates, and synthesizes information into findings.
- "minimal"     - anything else; a short summary plus next steps is enough.
Return the kind and a one-line reason."""

PLANNER_PROMPT = """You are the PLANNER in a self-refinement loop.
Turn the user's task into a short, ordered list of subgoals that are jointly sufficient to
finish it. Every subgoal needs a concrete 'done_when' acceptance criterion.
If you are given a CRITIQUE of a previous attempt, treat it as the top priority: restructure
the plan so the next attempt cannot repeat those faults. Keep the plan tight - 3 to 6 subgoals."""

WORKER_PROMPT = """You are the WORKER in a self-refinement loop.
Carry out the task by following the supplied plan. Use your tools to ground every factual
claim - do not assert anything you did not verify with a tool. Produce the best complete
answer you can, and respect any explicit limits in the task (length, format, scope).
If you are given a CRITIQUE checklist of what went wrong last time, you are starting FRESH -
do NOT assume any previous work exists - but make sure your new answer repeats none of the
listed faults."""

DEBATOR_PROMPT = """You are the DEBATOR: a relentless devil's advocate reviewing the WORKER's output.
Find everything wrong with it: unsupported or inaccurate claims, parts of the task it missed,
vague or unstructured sections, internal contradictions, limits it ignored, and anything the
handoff document overstates relative to what was actually produced.
Return one critique item per DISTINCT fault, each with a severity ('high'/'medium'/'low') and a
concrete fix hint. Be specific - name or quote the exact part you object to.
Do NOT invent faults to pad the list, and do NOT be charitable. If the answer is genuinely solid,
return an empty list of items."""

## 3. Tools & typed contracts

Two things live here: the **worker's tools** and the **typed contracts** the agents speak.

The worker gets two keyless tools over Wikipedia's public API — `search` and
`read_article` — so the notebook runs without a search-API key. These are the
"whatever tools" slot of the harness: swap them for filesystem tools, a code
runner, or a SQL client and the loop is unchanged. **The planner and debator get
no tools** — planning and critiquing are pure reasoning over text.

The contracts are Pydantic models passed as `output_schema`. The SDK returns the
validated instance in `reply.metadata["structured"]`. Two of them are load-bearing:
`Plan.subgoals` is what the worker executes, and `Critique.items` is what the loop
*counts* to decide whether to stop. Typing the critique is why the stop rule is a
clean `len(items) < DEBATOR_ITEMS` instead of a fragile bullet-parser.

> **Your turn:** the worker's only tools are `search` and `read_article`. If you
> pointed this loop at a *code-fix* task, which two tools would you add to the
> **worker**, and which single read-only tool would you give the **planner** so it
> stops planning blind to the codebase?

In [ ]:
import re
from vidbyte import tool

WIKI_API = "https://en.wikipedia.org/w/api.php"
HEADERS  = {"User-Agent": "vidbyte-cookbook-self-refine/0.1"}


@tool
def search(query: str) -> list[dict]:
    """Search Wikipedia for articles matching the query. Returns up to 5 results with
    title and a short snippet. Use read_article to read one in full."""
    resp = requests.get(WIKI_API, headers=HEADERS, timeout=30, params={
        "action": "query", "list": "search", "srsearch": query,
        "srlimit": 5, "format": "json",
    })
    resp.raise_for_status()
    return [
        {"title": h["title"], "snippet": re.sub(r"<[^>]+>", "", h["snippet"])}
        for h in resp.json()["query"]["search"]
    ]


@tool
def read_article(title: str) -> str:
    """Read the plain-text extract of one Wikipedia article by its exact title (truncated
    to 6 000 chars). Use the exact title returned by search()."""
    resp = requests.get(WIKI_API, headers=HEADERS, timeout=30, params={
        "action": "query", "prop": "extracts", "explaintext": 1,
        "titles": title, "format": "json", "redirects": 1,
    })
    resp.raise_for_status()
    pages = resp.json()["query"]["pages"]
    extract = next(iter(pages.values())).get("extract", "")
    return extract[:6_000] if extract else f"No article found for '{title}'."


WORKER_TOOLS = [search, read_article]


# ---- Typed stage contracts (output_schema) --------------------------------
class Subgoal(BaseModel):
    goal: str = Field(description="One concrete subgoal toward completing the task.")
    done_when: str = Field(description="Objective acceptance criterion for this subgoal.")


class Plan(BaseModel):
    summary: str = Field(description="One-sentence restatement of what the task must achieve.")
    subgoals: list[Subgoal] = Field(description="3-6 ordered, jointly sufficient subgoals.")


class CritiqueItem(BaseModel):
    issue: str = Field(description="Specifically what the worker got wrong.")
    severity: str = Field(description='"high", "medium", or "low".')
    fix_hint: str = Field(description="What a corrected attempt should do instead.")


class Critique(BaseModel):
    items: list[CritiqueItem] = Field(description="One entry per distinct fault; empty means no faults found.")
    verdict: str = Field(description="One-line overall assessment of the output.")


class HandoffChoice(BaseModel):
    kind: str = Field(description='Exactly one of "engineering", "research", "minimal".')
    reason: str = Field(description="One line on why this shape fits the task.")


def get_structured(reply, schema):
    """Pull the validated output_schema payload out of a reply, or raise if it failed to parse."""
    payload = reply.metadata.get("structured")
    if payload is None:
        raise RuntimeError(f"{schema.__name__} validation failed: {reply.content[:200]}")
    return payload if isinstance(payload, schema) else schema.model_validate(payload)


def render_plan(plan):
    """Render a Plan as a numbered checklist string for an agent prompt."""
    return "\n".join(f"{i}. {s.goal} (done when: {s.done_when})" for i, s in enumerate(plan.subgoals, 1))


def render_faults(critique):
    """Render a Critique as a severity-tagged bulleted list string."""
    if not critique.items:
        return "- (none)"
    return "\n".join(f"- [{c.severity}] {c.issue}  ->  fix: {c.fix_hint}" for c in critique.items)

## 4. Middleware

Middleware is deterministic runtime policy that the model never sees. A refinement
loop is exactly the place you want it: the worker runs its own tool loop, and the
*outer* loop multiplies every cost by the number of iterations. We attach budget +
runtime guards to the worker (the only agent that loops over tools):

| Middleware | Role in the refinement loop |
|---|---|
| `TokenBudgetMiddleware` | Hard token cap per worker attempt - one expensive attempt can't blow the whole run. |
| `CostBudgetMiddleware` | Dollar cap per worker attempt - converts token usage to dollars via `cost_per_million_tokens`. |
| `RuntimeLimitMiddleware` | Wall-clock + model-call cap - a worker stuck in a search loop can't hang the round. |

Each worker attempt gets a **fresh** set of these guards (new instances), so one round's
budget never bleeds into the next.

> **Your turn:** we cap each *worker attempt* with `CostBudgetMiddleware`, but the only
> thing bounding the *whole loop's* spend is `MAX_LOOP_ITERATIONS=3`. If one worker
> attempt can cost \$0.40, what's the rough worst-case dollar cost of a single
> `loop.run()` here — and where would you attach a guard that sees *all* rounds at once,
> rather than resetting every attempt?

In [ ]:
from vidbyte.middleware import (
    CostBudgetMiddleware,
    RuntimeLimitMiddleware,
    TokenBudgetMiddleware,
)

def build_worker_middleware():
    """Build a fresh set of budget + runtime guards for one worker attempt."""
    return [
        TokenBudgetMiddleware(max_tokens=TOKEN_BUDGET),
        CostBudgetMiddleware(max_spend_usd=MAX_COST_PER_AGENT, cost_per_million_tokens=COST_PER_MTOK),
        RuntimeLimitMiddleware(max_model_calls=15, max_elapsed_seconds=120.0),
    ]

## 5. The agents and the loop

Four agents, then the orchestrator that wires them.

- **classifier** — `output_schema=HandoffChoice`, run once.
- **planner** — `output_schema=Plan`, no tools. We **fork** it each round for fresh history.
- **make_worker(...)** — a *factory*, not a singleton. The handoff shape isn't known until
  the classifier runs, and `fork()` can't override an agent's `handoff=` (a fork always
  inherits its parent's handoff spec). So we build a fresh worker each round with the chosen
  shape — which also gives every attempt empty history.
- **debator** — `output_schema=Critique`, no tools; forked each round and handed the
  handoff via `context_items`.

The loop is a plain Python class, **not** a `SequentialPipeline`. Pipelines move
only strings between stages and can't loop or carry typed objects — but this loop
passes a `Plan` object forward, a `Handoff` object sideways into the debator, and a
`Critique` *backward* to the next planner. A class makes those edges explicit.

Two design choices worth pausing on:
1. **A fresh agent every round.** Planner and debator are *forked* (empty history); the
   worker is *rebuilt*. Either way, reusing one long-lived agent would let its old message
   history leak into the next attempt — defeating the "start fresh" property. A fresh agent
   means the critique is the *only* thing that crosses the round boundary.
2. **Handoff chosen once.** The task's nature doesn't change between rounds, so we
   classify the shape a single time and reuse it; re-classifying each round would
   cost a model call for no new signal.

> **Your turn:** every round builds a fresh worker, so only the explicit critique
> carries forward. If you instead reused one worker and kept its full history, what
> specific failure would you expect by round 3 — and how would that change what
> `DEBATOR_ITEMS` actually measures as a stop signal?

In [ ]:
from vidbyte import BaseAgent, AgentInput, EngineeringHandoff, ResearchHandoff, MinimalHandoff

HANDOFF_BY_KIND = {
    "engineering": EngineeringHandoff,
    "research": ResearchHandoff,
    "minimal": MinimalHandoff,
}

classifier = BaseAgent(
    name="handoff-classifier", system_prompt=CLASSIFIER_PROMPT,
    provider=PROVIDER, model_name=MODEL, output_schema=HandoffChoice,
)

planner = BaseAgent(
    name="planner", system_prompt=PLANNER_PROMPT,
    provider=PROVIDER, model_name=MODEL, output_schema=Plan,
)

def make_worker(handoff_spec, i):
    """Build a fresh worker (empty history, fresh guards) with the chosen handoff shape for round i."""
    return BaseAgent(
        name=f"worker-{i}", system_prompt=WORKER_PROMPT,
        provider=PROVIDER, model_name=MODEL, tools=WORKER_TOOLS,
        middleware=build_worker_middleware(), max_tool_rounds=MAX_TOOL_ROUNDS,
        handoff=handoff_spec,
    )


debator = BaseAgent(
    name="debator", system_prompt=DEBATOR_PROMPT,
    provider=PROVIDER, model_name=MODEL, output_schema=Critique,
)


@dataclass
class RefineResult:
    final_output: str        # the latest iteration's worker output (the returned answer)
    final_plan: Plan         # the plan that produced it
    stop_reason: str         # "converged" | "max_iterations"
    iterations_run: int
    history: list = field(default_factory=list)


class SelfRefineLoop:
    """Runs plan -> execute -> critique on a task until the debator runs out of faults or the cap."""

    def __init__(self, task, *, max_loop_iterations=MAX_LOOP_ITERATIONS, debator_items=DEBATOR_ITEMS):
        # Holds the task, the stop thresholds, the chosen handoff shape, and the run log.
        self.task = task
        self.max_loop_iterations = max_loop_iterations
        self.debator_items = debator_items
        self.handoff_spec = None
        self.history: list[dict] = []

    def run(self) -> RefineResult:
        """Pick the handoff shape once, then loop until a stop condition fires; return the latest output."""
        self.handoff_spec = self._choose_handoff()
        critique, plan, output = None, None, ""
        for i in range(1, self.max_loop_iterations + 1):
            plan = self._plan(critique, i)
            output, handoff_doc = self._execute(plan, critique, i)
            critique = self._critique(output, handoff_doc, i)
            self._record(i, plan, output, handoff_doc, critique)
            stop, reason = self._should_stop(critique, i)
            print(f"  iteration {i}: {len(critique.items)} fault(s) -> {reason or 'continue'}")
            if stop:
                return RefineResult(output, plan, reason, i, self.history)
        return RefineResult(output, plan, "max_iterations", self.max_loop_iterations, self.history)

    def _choose_handoff(self):
        """Classify the task once into a prebuilt handoff shape and reuse it every iteration."""
        choice = get_structured(classifier.run(self.task), HandoffChoice)
        spec_cls = HANDOFF_BY_KIND.get(choice.kind, MinimalHandoff)
        print(f"Handoff shape: {choice.kind} - {choice.reason}")
        return spec_cls()

    def _plan(self, critique, i) -> Plan:
        """Produce a fresh plan; on later rounds the latest critique is the priority input."""
        prompt = f"TASK:\n{self.task}"
        if critique is not None:
            prompt += "\n\nCRITIQUE OF THE PREVIOUS ATTEMPT (restructure to fix these):\n" + render_faults(critique)
        return get_structured(planner.fork(name=f"planner-{i}").run(prompt), Plan)

    def _execute(self, plan, critique, i):
        """Run a fresh worker against the plan (+ critique checklist) and return its output and handoff."""
        prompt = f"TASK:\n{self.task}\n\nPLAN:\n{render_plan(plan)}"
        if critique is not None:
            prompt += "\n\nDO NOT REPEAT THESE FAULTS FROM THE LAST ATTEMPT:\n" + render_faults(critique)
        worker = make_worker(self.handoff_spec, i)
        reply = worker.run(prompt)
        handoff_doc = reply.metadata.get("handoff") or worker.last_handoff
        if handoff_doc is None:
            print("  (note: handoff generation returned nothing; debator will review output text only)")
        return reply.content, handoff_doc

    def _critique(self, output, handoff_doc, i) -> Critique:
        """Have a fresh debator attack the output; pass the handoff as a context item when present."""
        prompt = f"TASK:\n{self.task}\n\nWORKER OUTPUT TO REVIEW:\n{output}"
        agent_input = AgentInput(prompt, context_items=(handoff_doc,)) if handoff_doc else prompt
        return get_structured(debator.fork(name=f"debator-{i}").run(agent_input), Critique)

    def _should_stop(self, critique, iteration):
        """Stop when the debator finds fewer than debator_items faults, or the iteration cap is hit."""
        if len(critique.items) < self.debator_items:
            return True, "converged"
        if iteration >= self.max_loop_iterations:
            return True, "max_iterations"
        return False, ""

    def _record(self, iteration, plan, output, handoff_doc, critique):
        """Append one iteration's artifacts to the in-memory run log."""
        self.history.append({
            "iteration": iteration, "plan": plan, "output": output,
            "handoff": handoff_doc, "critique": critique, "n_faults": len(critique.items),
        })

## 6. Running the loop

Instantiate `SelfRefineLoop` with a task and call `run()`. Watch the fault count
per round: a healthy run trends toward zero. The returned `RefineResult` carries
the latest answer plus the full per-round history so you can inspect exactly what
the debator caught and how the worker responded.

**Example tasks you can try** (each stresses the loop differently):

```
# Factual + length-limited (the debator catches inaccuracies and overruns)
"Write a tight, accurate 200-word explainer: why did the Hindenburg disaster
 effectively end the passenger-airship era? Ground every claim in sources you read."

# Comparison (the debator catches false symmetry and missing dimensions)
"Explain the practical differences between TCP and UDP for a backend engineer,
 in under 250 words, with a concrete example use case for each."

# Subjective (watch this one tend to run all the way to the iteration cap)
"Write three punchy one-line taglines for a note-taking app for researchers."
```

> **Your turn:** the loop stops on `len(items) < DEBATOR_ITEMS`. For the *subjective*
> tagline task, a devil's advocate can almost always find one more nit, so it burns
> all 3 iterations. What second stop signal that **isn't** the debator (think: a
> grader, a diff against the previous round, or a human gate) would you add so
> subjective tasks can terminate early?

In [ ]:
TASK = (
    "Write a tight, accurate 200-word explainer answering: why did the Hindenburg "
    "disaster effectively end the passenger-airship era? Ground every claim in sources "
    "you actually read, and stay at or under 200 words."
)

print("Running self-refinement loop...\n")
loop = SelfRefineLoop(TASK, max_loop_iterations=MAX_LOOP_ITERATIONS, debator_items=DEBATOR_ITEMS)
result = loop.run()

print(f"\nStopped after {result.iterations_run} iteration(s): {result.stop_reason}")
print("Faults per round:", [h["n_faults"] for h in result.history])

print("\nWhat the debator caught in round 1:")
print(render_faults(result.history[0]["critique"]))

print("\n" + "=" * 64 + "\nFINAL ANSWER (latest iteration)\n" + "=" * 64)
display(Markdown(result.final_output))

## 7. Example output

A representative run on the Hindenburg task. The fault count is the story — each
round the debator finds less to attack, until it converges:

```
Handoff shape: research - the task gathers and synthesizes facts into a short explainer.
  iteration 1: 3 fault(s) -> continue
  iteration 2: 1 fault(s) -> continue
  iteration 3: 0 fault(s) -> converged

Stopped after 3 iteration(s): converged
Faults per round: [3, 1, 0]
```

**What the debator caught in round 1:**

- `[high]` Claims the Hindenburg used hydrogen "because it was cheaper than helium" — reverses cause and effect; a US helium export ban forced the choice. → fix: state the embargo and that helium was US-controlled.
- `[medium]` The public-confidence claim has no source attached. → fix: cite the article actually read.
- `[low]` Explainer runs ~260 words, over the 200-word limit. → fix: tighten to ≤200 words.

**What the debator caught in round 2:**

- `[medium]` Asserts the disaster "ended the airship era" without connecting it to the actual shift to airplanes / cancelled programs. → fix: name the concrete consequence.

**Final answer (excerpt):**

> The 1937 Hindenburg fire destroyed the airship in under a minute, killing 36
> people on a routine transatlantic crossing (Hindenburg disaster, Wikipedia). The
> craft was lifted by flammable hydrogen because the United States — then the only
> large producer of non-flammable helium — barred its export under the 1927 Helium
> Control Act (Helium, Wikipedia)... Filmed and broadcast worldwide, the disaster
> collapsed public confidence in passenger airships just as faster, cheaper
> airplanes were maturing, and commercial rigid-airship service never recovered
> (Hindenburg disaster, Wikipedia).

---

### What a production self-refinement system adds

- **Return the best round, not the latest.** Track the lowest-fault output and return
  that — robust when the loop stops on the iteration cap rather than convergence.
  (One line here: `min(result.history, key=lambda h: h["n_faults"])`.)
- **Accumulating reflection memory.** Real Reflexion keeps *every* past critique in
  the worker's memory, not just the latest one — at the cost of growing context.
- **A non-LLM stop signal.** Pair the debator with a deterministic grader (tests, a
  JSON-schema check, a word-count assert) so a sycophantic *or* nitpicking critic
  can't single-handedly decide termination.
- **Ensemble debators.** Run several debators and merge/vote on faults to cut the
  variance of a single critic.
- **Full-loop tracing.** Wrap the run in `Trace`/continual tracing to replay how the
  answer evolved across rounds.

### Things to try

1. Set `DEBATOR_ITEMS = 2` and rerun — the loop tolerates one lingering low-severity
   fault and stops a round earlier.
2. Swap `WORKER_TOOLS` for filesystem/code tools and give it a code-fix task; the
   loop structure doesn't change, only the worker's tools and the handoff shape.
3. Switch `VIDBYTE_COOKBOOK_PROVIDER=anthropic` and `VIDBYTE_COOKBOOK_MODEL=claude-opus-4-8`
   in `.env` and compare how many rounds each model needs to converge.